# Handoffs

Season 2 starts the Hidden Leaf Village Mission Ops Center: a Mission Desk agent
that routes requests to specialist ninja agents instead of doing everything
itself. This notebook covers `handoff()`, `on_handoff`, `input_filter`, and
`nest_handoff_history`.


## A basic handoff

`handoff(agent)` wraps an `Agent` as something another agent can hand off to;
the model triggers it via a `transfer_to_<agent_name>`-named tool by default.
`prompt_with_handoff_instructions(...)` primes the Mission Desk's instructions
with the SDK's recommended multi-agent-system context.


In [14]:
from typing import Any  # the type for extra handoff() options passed straight through

from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    FunctionTool,  # the tool object that @function_tool makes from a normal function
    handoff,  # wraps an Agent so another agent can pass the chat to it
)
from agents.extensions.handoff_prompt import (
    RECOMMENDED_PROMPT_PREFIX,  # a ready-made text the SDK suggests for agents that hand off
    prompt_with_handoff_instructions,  # adds that ready-made text in front of your own instructions
)

# Initialize the specialist agent — used by every demo below
mission_response_team = Agent(
    name="Mission Response Team",
    instructions="You handle mission emergencies — dispatch and confirm.",
)


# Test helper: builds a Mission Desk that hands off to the specialist. Every demo below
# calls it, and extra handoff() options (on_handoff, input_filter) pass straight through
def build_desk(
    instructions: str = "Route mission emergencies.",  # what the desk is told to do
    tools: list[FunctionTool] | None = None,  # tools the desk can call before it hands off
    **handoff_options: Any,  # extra handoff() options, like on_handoff or input_filter
) -> Agent:
    # Initialize the mission desk agent
    return Agent(
        name="Village Mission Desk",
        instructions=instructions,
        # tools: the tools this agent is allowed to call
        tools=tools or [],
        # handoffs: other agents this one can pass the conversation to
        handoffs=[handoff(mission_response_team, **handoff_options)],
    )


mission_desk = build_desk(
    prompt_with_handoff_instructions(
        "Route mission emergencies to the Mission Response Team. Handle everything else yourself."
    )
)

print("[handoff tool name]", mission_desk.handoffs[0].tool_name)
print("[prompt prefix]", RECOMMENDED_PROMPT_PREFIX[:80], "...")

[handoff tool name] transfer_to_mission_response_team
[prompt prefix] # System context
You are part of a multi-agent system called the Agents SDK, des ...


## Run it, offline

Script the model to call the handoff tool, then have the now-active specialist
answer. `result.last_agent` switches to the specialist.


In [15]:
from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
    function_call,  # makes one scripted tool-call step for ScriptedModel
)


# Demo: tells the model to use the handoff tool, then shows that control
# really moved to the specialist: last_agent changes and new_items records the handoff
async def demo_handoff() -> None:
    # Run the Agent
    result = await Runner.run(
        mission_desk,
        "There's a mission emergency at Mission-07!",
        # run_config: settings for just this run
        run_config=RunConfig(
            # model: use this model for this run only
            model=ScriptedModel(
                [
                    [function_call("transfer_to_mission_response_team", {}, call_id="c1")],
                    [assistant_message("Mission Response Team dispatched to Mission-07.")],
                ]
            ),
            # workflow_name: the name this run shows up under in tracing
            workflow_name="e08-handoff-demo",
        ),
    )
    # proves the handoff really moved control: last_agent is now the
    # specialist, and new_items records the handoff call and result next to the message
    print("[final_output]", result.final_output)
    print("[last_agent]", result.last_agent.name)
    print("[new_items]", [type(item).__name__ for item in result.new_items])


await demo_handoff()
# Outside a notebook: asyncio.run(demo_handoff())

[final_output] Mission Response Team dispatched to Mission-07.
[last_agent] Mission Response Team
[new_items] ['HandoffCallItem', 'HandoffOutputItem', 'MessageOutputItem']


## `on_handoff` — side effects

Runs whenever the handoff fires — useful for logging or bookkeeping. It never
chooses the destination.


In [16]:
from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    RunContextWrapper,  # holds the context= value you passed, and gives it to callbacks
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
    function_call,  # makes one scripted tool-call step for ScriptedModel
)

handoff_log: list[str] = []


# on_handoff callback: an extra action that runs when the handoff happens. It is
# connected with build_desk(on_handoff=log_handoff) below
def log_handoff(
    ctx: RunContextWrapper,  # holds this run's context=, whatever you passed
) -> None:
    handoff_log.append(ctx.context)


# Demo: adds on_handoff to a new desk and shows it runs as an extra action
# during the run, without changing which specialist is chosen
async def demo_on_handoff() -> None:
    # Run the Agent
    await Runner.run(
        build_desk(on_handoff=log_handoff),
        "Mission emergency!",
        # context: your own value that tools and callbacks can read during the run
        context="caller: Shikamaru Nara",
        # run_config: settings for just this run
        run_config=RunConfig(
            # model: use this model for this run only
            model=ScriptedModel(
                [
                    [function_call("transfer_to_mission_response_team", {}, call_id="c1")],
                    [assistant_message("done")],
                ]
            ),
            # workflow_name: the name this run shows up under in tracing
            workflow_name="e08-on-handoff-demo",
        ),
    )
    # proves on_handoff really ran as an extra action during the run, and got
    # the same context= that was passed into Runner.run
    print("[handoff_log]", handoff_log)


await demo_on_handoff()
# Outside a notebook: asyncio.run(demo_on_handoff())

[handoff_log] ['caller: Shikamaru Nara']


## `input_filter` — controlling what the next agent sees

By default the specialist sees the entire prior transcript, including the
Mission Desk's own tool calls. `remove_all_tools` strips those out — proven here
by capturing the exact input the specialist's model call receives.


In [17]:
from typing import Any  # the type for an input_filter, which can be a function or None

from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
    function_tool,  # turns a normal function into a tool an Agent can call
)
from agents.extensions.handoff_filters import (
    remove_all_tools,  # an input_filter that removes tool calls and tool results
)
from agents.testing import (
    ModelStep,  # one scripted step, made from your own function
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
    function_call,  # makes one scripted tool-call step for ScriptedModel
)


# Define the tool
# @function_tool callback: gives the Mission Desk something to call before the handoff,
# so there is a real tool-call item for input_filter to remove below
@function_tool
def check_mission_status(mission_id: str) -> str:
    """Check a mission's danger level (stubbed for this demo)."""
    return "danger level 9, evacuate"


captured_inputs: list[list] = []


# Test helper: short labels for input items, used by the demos in this and the next cell
def input_types(items: list) -> list[str]:
    # Messages have a "role", and everything else has a "type"
    return [i.get("role", i.get("type")) for i in items]


# ModelStep responder: acts like the specialist's real model call so we can
# catch the exact input it got. It is used in ModelStep(responder=capture_and_answer)
async def capture_and_answer(call) -> ModelStep:  # call: the model request that is running now
    captured_inputs.append(call.input)
    return ModelStep(output=[assistant_message("Handled by specialist.")])


# Test helper: runs one handoff with the given settings and gives back what the specialist's
# model call really saw. The demos below call it with different input_filter / nest values
async def run_and_capture(input_filter: Any = None, nest: bool = False) -> list[str]:
    captured_inputs.clear()
    # Script the model
    scripted = ScriptedModel(
        [
            [function_call("check_mission_status", {"mission_id": "Mission-07"}, call_id="c1")],
            [function_call("transfer_to_mission_response_team", {}, call_id="c2")],
            ModelStep(responder=capture_and_answer),
        ]
    )
    # Run the Agent
    await Runner.run(
        build_desk(
            # tools: tools the desk can call before it hands off
            tools=[check_mission_status],
            # input_filter: changes the chat history the next agent gets to see
            input_filter=input_filter,
        ),
        "emergency at Mission-07",
        # run_config: settings for just this run
        run_config=RunConfig(
            # model: use this model for this run only
            model=scripted,
            # workflow_name: the name this run shows up under in tracing
            workflow_name="e08-filter-demo",
            # nest_handoff_history: squeezes the earlier chat into one message
            nest_handoff_history=nest,
        ),
    )
    return input_types(captured_inputs[0])


# Demo: calls run_and_capture with no filter, then with remove_all_tools, to show
# how what the specialist sees changes between the two
async def demo_input_filter() -> None:
    # proves the specialist sees the Mission Desk's own tool-call items by default
    print("[without input_filter]", await run_and_capture())
    # proves remove_all_tools removes those tool-call and tool-result items before the
    # specialist's model call sees them
    print("[with remove_all_tools]", await run_and_capture(remove_all_tools))


await demo_input_filter()
# Outside a notebook: asyncio.run(demo_input_filter())

[without input_filter] ['user', 'function_call', 'function_call_output', 'function_call', 'function_call_output']
[with remove_all_tools] ['user']


## `nest_handoff_history` — opt-in transcript summarization

`RunConfig(nest_handoff_history=True)` collapses the whole prior transcript
into one summarized message — disabled by default, confirmed by comparing both
settings.


In [18]:
# Demo: calls run_and_capture (from the cell above) with nest_handoff_history off, then on,
# to show the earlier chat the specialist sees shrink into one message
async def demo_nest_handoff_history() -> None:
    default_types = await run_and_capture()
    nested_types = await run_and_capture(nest=True)
    # proves nest_handoff_history is off by default (the full chat is kept)
    print(f"[nest_handoff_history=False] {len(default_types)} items: {default_types}")
    # proves turning it on squeezes that same chat into one short summary message
    print(f"[nest_handoff_history=True]  {len(nested_types)} items: {nested_types}")


await demo_nest_handoff_history()
# Outside a notebook: asyncio.run(demo_nest_handoff_history())

[nest_handoff_history=False] 5 items: ['user', 'function_call', 'function_call_output', 'function_call', 'function_call_output']
[nest_handoff_history=True]  1 items: ['assistant']


## Run it live — optional, needs a real key

Three real runs of the same Mission Desk → Mission Response Team handoff,
mirroring the scripted demos above. Each run has `on_handoff` wired in and a
`check_mission_status` tool the desk is told to call first. The first run uses
no filter, the second uses `input_filter=remove_all_tools`, and the third uses
`RunConfig(nest_handoff_history=True)`. There's no `ScriptedModel` here to
record `.calls`, so `RunConfig(call_model_input_filter=...)` captures the exact
input the specialist's real model call receives. That's the live equivalent of
`capture_and_answer` above. The live model decides whether to hand off at all.
A clear emergency plus explicit routing instructions makes a handoff very
likely, but the cell prints what happened on each run either way rather than
assuming it.


In [19]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.extensions.handoff_filters import (
    remove_all_tools,  # an input_filter that removes tool calls and tool results
)
from agents.extensions.handoff_prompt import (
    prompt_with_handoff_instructions,  # adds that ready-made text in front of your own instructions
)
from agents.run_config import (
    CallModelData,  # the info a model-input filter gets: the model data, the agent, and the context
    ModelInputData,  # the instructions and input that are about to go to the model
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# call_model_input_filter callback: saves what the specialist's real model call is about
# to get, then passes it on unchanged. It is connected with RunConfig(call_model_input_filter=...)
def capture_specialist_input(
    data: CallModelData,  # the model call that is about to happen: model_data, agent, context
) -> ModelInputData:
    if data.agent.name == mission_response_team.name:
        captured_inputs.append(data.model_data.input)
    return data.model_data


# Demo: three real handoff runs like the scripted demos above, but with the live model,
# reusing build_desk, check_mission_status, log_handoff and captured_inputs from earlier cells
async def demo_live() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    runs = [
        ("without input_filter", None, False),
        ("with remove_all_tools", remove_all_tools, False),
        ("nest_handoff_history=True", None, True),
    ]
    for label, input_filter, nest in runs:
        captured_inputs.clear()
        handoff_log.clear()
        # Run the Agent
        live_result = await Runner.run(
            build_desk(
                prompt_with_handoff_instructions(
                    "For any mission emergency: first call check_mission_status for that mission, "
                    "then transfer to the Mission Response Team. Handle everything else yourself."
                ),
                # tools: tools the desk can call before it hands off
                tools=[check_mission_status],
                # on_handoff: a function that runs when the handoff happens
                on_handoff=log_handoff,
                # input_filter: changes the chat history the next agent gets to see
                input_filter=input_filter,
            ),
            "There's a mission emergency at Mission-12, enemy ninjas spotted!",
            # context: your own value that tools and callbacks can read during the run
            context="caller: Shikamaru Nara",
            # run_config: settings for just this run
            run_config=RunConfig(
                # workflow_name: the name this run shows up under in tracing
                workflow_name=f"e08-live-{label}",
                # nest_handoff_history: whether the earlier chat is squeezed into one message at a handoff
                nest_handoff_history=nest,
                # call_model_input_filter: the live version of ScriptedModel's .calls
                call_model_input_filter=capture_specialist_input,
            ),
        )

        # Inspect the result
        # proves last_agent changes to the specialist when the live model hands off
        print(f"[{label}] last_agent={live_result.last_agent.name}")
        if captured_inputs:
            # proves what the specialist really saw: tool items by default, none after
            # remove_all_tools, and one squeezed message with nest_handoff_history
            seen = input_types(captured_inputs[0])
            print(f"[{label}] specialist saw {len(seen)} items: {seen}")
        else:
            print(f"[{label}] the live model answered without handing off this time")
        # proves on_handoff ran once for the real handoff, and got the run's context=
        print(f"[{label}] on_handoff log: {handoff_log}")
        print("-" * 50)


await demo_live()
# Outside a notebook: asyncio.run(demo_live())

[without input_filter] last_agent=Mission Response Team
[without input_filter] specialist saw 5 items: ['user', 'function_call', 'function_call_output', 'function_call', 'function_call_output']
[without input_filter] on_handoff log: ['caller: Shikamaru Nara']
--------------------------------------------------
[with remove_all_tools] last_agent=Mission Response Team
[with remove_all_tools] specialist saw 1 items: ['user']
[with remove_all_tools] on_handoff log: ['caller: Shikamaru Nara']
--------------------------------------------------
[nest_handoff_history=True] last_agent=Mission Response Team
[nest_handoff_history=True] specialist saw 1 items: ['assistant']
[nest_handoff_history=True] on_handoff log: ['caller: Shikamaru Nara']
--------------------------------------------------


## Summary

- `handoff(agent)` wraps an `Agent` as a handoff target, triggered via a
  `transfer_to_<agent_name>`-named tool by default; `on_handoff` runs side
  effects but never changes the destination.
- `input_filter` controls what the next agent sees (`remove_all_tools` strips
  tool items), and `nest_handoff_history` (opt-in, off by default) collapses the
  whole prior transcript into one summarized message.
